In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "/Users/xinc/GitHub/note")
sys.path.insert(0, "/Users/xinc/GitHub/Quant")
# Local project path is optional; notebook directory is already on sys.path.
# sys.path.insert(0, "D:/Github/Quant/projects/CTA/學弟策略")

from pathlib import Path
import gc
import time
from datetime import datetime
import pandas as pd
from tqdm import tqdm

from module.get_info_FinMind import FinMindClient
from utils import (
    chunk_list,
    compact_daily_price_parquet,
    get_valid_stock_ids,
    get_missing_kbar_ids,
    infer_stock_ids_from_kbar_dir,
    load_no_price_pairs,
    load_or_build_above_ma_signal,
    load_or_refresh_daily_prices,
    read_kbar_ids,
    read_kbar_with_supplement,
    build_kbar_coverage_report,
    summarize_kbar_coverage,
    signal_row_to_stock_ids,
    update_missing_kbar_ids,
    update_missing_full_kbar_dates,
)

client = FinMindClient()
from cloud_data import (
    DATA_ROOT,
    TRADING_ROOT,
    TW_STOCK_DAILY_PRICE,
    TW_STOCK_KBAR_1MIN,
    TW_STOCK_KBAR_ABOVE_MA60,
    TW_FUTURES_TX,
)

2026-07-14 15:30:19.129 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


## get data

### config

In [5]:
start_date = "2022-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

ma_window = 60
retry_wait = 60
max_retries = 1
rate_limit_wait = 3660
rate_limit_max_retries = 2
api_batch_size = 30
rebuild_signal = True

# Fetch extra daily history so the first target date can have a valid MA60.
daily_start_date = (
    pd.Timestamp(start_date) - pd.Timedelta(days=ma_window * 3)
).strftime("%Y-%m-%d")

data_dir = TRADING_ROOT
stock_universe_output = DATA_ROOT / "trading" / "market_reference" / "note_data" / "stock_universe.parquet"
daily_output = TW_STOCK_DAILY_PRICE
full_kbar_dir = TW_STOCK_KBAR_1MIN
signal_output = full_kbar_dir / "above_ma60_signal.parquet"
kbar_output_dir = TW_STOCK_KBAR_ABOVE_MA60
missing_kbar_output = kbar_output_dir / "missing_kbar_ids.json"
missing_full_kbar_output = kbar_output_dir / "missing_full_kbar_dates.json"
kbar_output_dir.mkdir(parents=True, exist_ok=True)

### get daily data

In [6]:
# 1. Load or refresh raw daily prices for valid stocks only.
stock_ids = get_valid_stock_ids(client, output_file=stock_universe_output)
print(f"Valid stock count: {len(stock_ids):,}")

daily = load_or_refresh_daily_prices(
    client=client,
    output_file=daily_output,
    start_date=daily_start_date,
    end_date=end_date,
    stock_ids=stock_ids,
    retry_wait=retry_wait,
)

daily["date"] = pd.to_datetime(daily["date"])
print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")

Valid stock count: 2,138


Download daily price: 100%|██████████| 30/30 [04:14<00:00,  8.48s/day]


Daily rows: 2,329,905
Daily dates: 1,220
Daily stocks: 2,052
Daily range: 2021-07-05 ~ 2026-07-13


In [17]:
# Optional: compact the old huge daily_stock_price.parquet without calling API.
old_daily_output = data_dir / "daily_stock_price.parquet"

if not old_daily_output.exists():
    raise FileNotFoundError(old_daily_output)

if stock_universe_output.exists():
    stock_ids_for_compact = pd.read_parquet(stock_universe_output)["stock_id"].astype(str)
    stock_ids_for_compact = stock_ids_for_compact[stock_ids_for_compact.str.fullmatch(r"[1-9]\d{3}")].tolist()
else:
    stock_ids_for_compact = infer_stock_ids_from_kbar_dir(full_kbar_dir)
    if not stock_ids_for_compact:
        raise FileNotFoundError(
            "No stock_universe.parquet and no kbar/1min parquet files to infer stock ids from."
        )
    pd.DataFrame({"stock_id": stock_ids_for_compact}).to_parquet(
        stock_universe_output,
        index=False,
    )

print(f"Compacting {len(stock_ids_for_compact):,} stock ids")

daily = compact_daily_price_parquet(
    source_file=old_daily_output,
    output_file=daily_output,
    stock_ids=stock_ids_for_compact,
)

print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")

Compacting 2,138 stock ids


Compact daily price: 100%|██████████| 5/5 [00:00<00:00, 10.62batch/s]


Daily rows: 2,272,554
Daily dates: 1,191
Daily stocks: 2,022
Daily range: 2021-07-05 ~ 2026-05-29


### build 60ma signal

In [7]:
# 2. Build/load date x stock_id signal matrix.
# True means this stock passed yesterday's close > MA60 condition for today's kbar.
if "daily" not in globals():
    daily = pd.read_parquet(
        daily_output,
        columns=["date", "stock_id", "close"],
    )

signal = load_or_build_above_ma_signal(
    daily=daily,
    output_file=signal_output,
    start_date=start_date,
    end_date=end_date,
    ma_window=ma_window,
    rebuild=rebuild_signal,
)

print(f"Signal shape: {signal.shape[0]} dates x {signal.shape[1]} stocks")
print(f"Signal true cells: {int(signal.to_numpy().sum())}")

del daily
gc.collect()

Signal shape: 1094 dates x 2052 stocks
Signal true cells: 949415


0

### get minute kbar data

In [ ]:
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)

no_price_pairs = load_no_price_pairs(daily_output)

# 3. Build supplemental K bars. Full kbar/1min is primary; above_ma60 stores only missing symbols needed for this study.
signal_desc = signal.sort_index(ascending=False)

bar = tqdm(signal_desc.iterrows(), total=len(signal_desc), desc="Build filtered kbar")
for trade_date, signal_row in bar:
    stock_ids = signal_row_to_stock_ids(signal_row)
    date_str = pd.Timestamp(trade_date).strftime("%Y-%m-%d")
    bar.set_postfix_str(f"{date_str} check")

    if not stock_ids:
        bar.set_postfix_str(f"{date_str} empty")
        continue

    full_file = full_kbar_dir / f"{date_str}.parquet"
    supplement_file = kbar_output_dir / f"{date_str}.parquet"

    if full_file.exists():
        full_ids = read_kbar_ids(full_file)
    else:
        # Full-day kbar is unavailable; still fetch the strategy-required symbols
        # into above_ma60 so the study can run without a complete full kbar file.
        update_missing_full_kbar_dates(missing_full_kbar_output, date_str)
        full_ids = set()
        bar.set_postfix_str(f"{date_str} full missing; supplement")

    known_missing_ids = get_missing_kbar_ids(missing_kbar_output, date_str)

    expected_ids = set(stock_ids) - known_missing_ids
    if not expected_ids:
        bar.set_postfix_str(f"{date_str} all missing-known")
        continue

    supplement_ids = read_kbar_ids(supplement_file)
    available_ids = full_ids | supplement_ids

    api_ids = expected_ids - available_ids
    if not api_ids:
        bar.set_postfix_str(f"{date_str} covered")
        continue

    no_price_ids = {
        stock_id for stock_id in api_ids
        if (date_str, stock_id) in no_price_pairs
    }
    if no_price_ids:
        update_missing_kbar_ids(missing_kbar_output, date_str, no_price_ids)
        api_ids -= no_price_ids
        bar.set_postfix_str(f"{date_str} no price {len(no_price_ids)}")
        if not api_ids:
            continue

    api_batches = chunk_list(sorted(api_ids), api_batch_size)
    for batch_idx, api_batch in enumerate(api_batches, start=1):
        bar.set_postfix_str(
            f"{date_str} api {batch_idx}/{len(api_batches)} ({len(api_batch)})"
        )
        try:
            report = client.get_multi_kbar(
                start_date=date_str,
                end_date=date_str,
                output_dir=kbar_output_dir,
                stock_ids=api_batch,
                max_retries=1,
                retry_wait=retry_wait,
                rate_limit_wait=rate_limit_wait,
                rate_limit_max_retries=rate_limit_max_retries,
                return_report=True,
                verbose=False,
                show_progress=False,
                status_callback=lambda msg: bar.set_postfix_str(msg),
            )
            if report is False or report.get("rate_limited"):
                bar.set_postfix_str(f"stopped {date_str}")
                print(f"Stopped at {date_str}; rerun this cell later to resume.")
                break

            missing_from_download = report.get("missing_stock_ids_by_date", {}).get(date_str, [])
            if missing_from_download:
                update_missing_kbar_ids(missing_kbar_output, date_str, missing_from_download)
                bar.set_postfix_str(f"{date_str} missing {len(missing_from_download)}")
            else:
                bar.set_postfix_str(f"{date_str} api batch done")
        except Exception as exc:
            bar.set_postfix_str(f"{date_str} failed: {type(exc).__name__}")
            print(f"{date_str} failed: {type(exc).__name__}: {exc}")
            time.sleep(retry_wait)
    else:
        continue
    break

Build filtered kbar:  90%|████████▉ | 980/1094 [13:38:20<1:35:11, 50.10s/it, 2022-06-27 rate limit 1/2 wait 3660s]  


KeyboardInterrupt: 

: 

In [ ]:
report = client.get_multi_kbar(
            start_date='2026-04-06',
            end_date='2026-04-06',
            output_dir=kbar_output_dir,
            stock_ids=[],
            max_retries=1,
            retry_wait=retry_wait,
            rate_limit_wait=rate_limit_wait,
            rate_limit_max_retries=rate_limit_max_retries,
            return_report=True,
            verbose=True,
            show_progress=False,
        )

股票數: 1
交易日數: 1
已完成: 0 天 / 待下載: 1 天
預估剩餘: 0.0 小時


In [3]:
# Optional inspection
# signal = pd.read_parquet(TW_STOCK_KBAR_1MIN / "2025-03-05.parquet")
signal = pd.read_parquet(TW_STOCK_KBAR_1MIN / "2021-01-07.parquet")
signal

,date,minute,stock_id,open,high,low,close,volume
0,2021-01-07,09:00:00,2330,554.0,555.0,554.0,555.0,2636
1,2021-01-07,09:01:00,2330,555.0,555.0,553.0,554.0,482
2,2021-01-07,09:02:00,2330,554.0,555.0,553.0,554.0,358
3,2021-01-07,09:03:00,2330,555.0,555.0,554.0,554.0,221
4,2021-01-07,09:04:00,2330,555.0,556.0,554.0,556.0,532
...,...,...,...,...,...,...,...,...
527,2021-01-07,13:21:00,2317,106.5,107.0,106.5,106.5,458
528,2021-01-07,13:22:00,2317,107.0,107.0,106.5,106.5,541
529,2021-01-07,13:23:00,2317,107.0,107.0,106.5,107.0,663
530,2021-01-07,13:24:00,2317,107.0,107.0,106.5,107.0,3152


### inspection

In [ ]:
# Check whether the MA60 signal universe is covered by full kbar + supplement files.
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)

coverage = build_kbar_coverage_report(
    signal=signal,
    full_kbar_dir=full_kbar_dir,
    supplement_dir=kbar_output_dir,
)
missing_full_top, incomplete_full_top = summarize_kbar_coverage(coverage, top_n=30)

missing_full_count = int((~coverage["full_exists"]).sum())
incomplete_full_count = int((coverage["full_exists"] & (coverage["missing_count"] > 0)).sum())

print(f"Coverage dates: {len(coverage):,}")
print(f"Missing full kbar dates: {missing_full_count:,}")
print(f"Full exists but signal still missing kbar: {incomplete_full_count:,}")

print("\nTop missing full kbar dates:")
display(missing_full_top)

print("\nTop incomplete full kbar dates:")
display(incomplete_full_top)

# Inspect one date by setting inspect_date, for example: inspect_date = "2025-02-14"
inspect_date = None
if inspect_date is not None:
    inspect_row = coverage.loc[coverage["date"] == inspect_date].iloc[0]
    inspect_missing_ids = inspect_row["missing_ids"]
    print(f"{inspect_date} missing ids: {len(inspect_missing_ids):,}")
    print(inspect_missing_ids[:100])

Coverage dates: 1,094
Missing full kbar dates: 942
Full exists but signal still missing kbar: 120

Top missing full kbar dates:


,date,signal_count,full_exists,full_count,supplement_count,covered_count,missing_count
1093,2026-07-13,941,False,0,941,941,0
1092,2026-07-09,983,False,0,972,972,11
1091,2026-07-08,1015,False,0,1004,1004,11
1090,2026-07-07,1204,False,0,1204,1204,0
1089,2026-07-06,1193,False,0,1193,1193,0
1088,2026-07-03,1052,False,0,1052,1052,0
1087,2026-07-02,968,False,0,968,968,0
1086,2026-07-01,978,False,0,978,978,0
1085,2026-06-30,898,False,0,898,898,0
1084,2026-06-29,829,False,0,829,829,0



Top incomplete full kbar dates:


,date,signal_count,full_exists,full_count,supplement_count,covered_count,missing_count
764,2025-03-06,1246,True,2179,0,1194,52
763,2025-03-05,1165,True,2177,0,1116,49
594,2024-06-19,1267,True,2128,0,1219,48
836,2025-06-20,762,True,2210,0,715,47
751,2025-02-14,1040,True,106,937,993,47
885,2025-08-28,1017,True,2229,0,970,47
691,2024-11-12,762,True,2144,0,715,47
860,2025-07-24,752,True,2216,0,706,46
861,2025-07-25,752,True,2214,0,706,46
884,2025-08-27,971,True,2240,0,925,46


## backtest